# EEG-CogState — Explicabilité IA (XAI) avec SHAP

**Objectif** : comprendre **pourquoi** le modèle prend ses décisions, au lieu de se contenter de prédire. C'est l'IA explicable — ce qui distingue un projet qui *applique* le ML d'un projet qui le *comprend*.

## Qu'est-ce que SHAP ?

SHAP (SHapley Additive exPlanations) attribue à chaque caractéristique une **contribution** : combien elle a poussé la décision vers une classe. Issu de la théorie des jeux (valeurs de Shapley), c'est la méthode de référence en explicabilité.

**Prérequis** : avoir lancé le Sprint 3 (`features.csv`) et le Sprint 4 (`model.joblib`).

## 1. Chargement du modèle et des données

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np
import pandas as pd
import joblib
import matplotlib.pyplot as plt

from src import config, explainability

df = pd.read_csv(config.PROCESSED_DIR / "features.csv")
meta = ["subject", "label"]
feature_names = [c for c in df.columns if c not in meta]
X = df[feature_names].values
model = joblib.load(config.MODELS_DIR / "model.joblib")

print(f"{X.shape[0]} epochs, {X.shape[1]} features")
print("Classes :", list(model.classes_))

## 2. Calcul des valeurs SHAP

On utilise `TreeExplainer`, exact et rapide pour les forêts d'arbres.

> Peut prendre quelques dizaines de secondes.

In [ ]:
shap_result = explainability.compute_shap_values(model, X, feature_names)
print("Valeurs SHAP calculées :", shap_result["shap_matrix"].shape,
      "(échantillons × features)")

## 3. Importance par bande de fréquence

L'interprétation la plus parlante : quelles **bandes cérébrales** le modèle
utilise-t-il le plus ? On attend alpha et bêta en tête (relaxation vs concentration).

In [ ]:
bands = explainability.importance_by_band(shap_result)

band_colors = {"delta": "#85B7EB", "theta": "#5DCAA5", "alpha": "#1D9E75",
               "beta": "#EF9F27", "gamma": "#E24B4A"}
colors = [band_colors.get(b, "#888") for b in bands["band"]]

fig, ax = plt.subplots(figsize=(8, 4))
ax.bar(bands["band"], bands["importance"], color=colors)
ax.set_xlabel("Bande de fréquence")
ax.set_ylabel("Importance cumulée (|SHAP|)")
ax.set_title("Importance des bandes EEG dans les décisions du modèle")
ax.grid(True, alpha=0.3, axis="y")
plt.show()

print(bands.to_string(index=False))

## 4. Importance par canal (électrode)

Quelles **électrodes** sont les plus déterminantes ? Cela localise l'information
discriminante sur le scalp.

In [ ]:
chans = explainability.importance_by_channel(shap_result, top=10)

fig, ax = plt.subplots(figsize=(8, 4))
ax.bar(chans["channel"], chans["importance"], color="#2E75B6")
ax.set_xlabel("Canal")
ax.set_ylabel("Importance cumulée (|SHAP|)")
ax.set_title("Importance des canaux EEG")
ax.grid(True, alpha=0.3, axis="y")
plt.show()

print(chans.to_string(index=False))

## 5. Features les plus influentes (globalement)

Le détail : les caractéristiques individuelles qui pèsent le plus.

In [ ]:
top = explainability.global_feature_importance(shap_result, top=15)

fig, ax = plt.subplots(figsize=(9, 5))
ax.barh(top["feature"][::-1], top["importance"][::-1], color="#1D9E75")
ax.set_xlabel("Importance (|SHAP| moyen)")
ax.set_title("15 features les plus influentes")
ax.grid(True, alpha=0.3, axis="x")
plt.tight_layout()
plt.show()

print(top.to_string(index=False))

## 6. Expliquer UNE prédiction précise

Pour un epoch donné, quelles features ont poussé vers la classe prédite ?
Ici le **signe** compte : positif = en faveur de la classe prédite.

In [ ]:
idx = 0  # on explique le premier epoch (change l'index pour en voir d'autres)
predicted = model.predict(X[idx:idx+1])[0]
print(f"Epoch #{idx} — classe prédite : {predicted}\n")

expl = explainability.explain_single_prediction(model, X[idx], feature_names, top=10)

colors = ["#1D9E75" if v > 0 else "#E24B4A" for v in expl["contribution"]]
fig, ax = plt.subplots(figsize=(9, 5))
ax.barh(expl["feature"][::-1], expl["contribution"][::-1],
        color=colors[::-1])
ax.axvline(0, color="#888", linewidth=0.8)
ax.set_xlabel("Contribution SHAP (+ vers la classe prédite)")
ax.set_title(f"Pourquoi le modèle prédit '{predicted}' pour cet epoch")
ax.grid(True, alpha=0.3, axis="x")
plt.tight_layout()
plt.show()

print(expl.to_string(index=False))

## ✅ Bilan — IA explicable

On sait désormais expliquer le modèle à trois niveaux :
- **par bande** (quelles ondes cérébrales comptent) ;
- **par canal** (quelles électrodes) ;
- **par feature** et **pour une prédiction précise** (avec le signe).

C'est un atout fort : le modèle n'est plus une boîte noire. Si alpha et bêta
ressortent en tête, c'est **cohérent avec la neurophysiologie** — preuve que
le modèle a appris des marqueurs sensés, pas du bruit.